Importy

In [16]:
import pandas as pd
from prophet import Prophet
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "browser"


Wczytanie danych

In [2]:
df = pd.read_csv("orders_with_products.csv", sep=";") # plik polaczonych tabel z excela

# zamiana stringa na datetime, dayfirst=True, format to DD.MM.YYYY
df["order_purchase_timestamp"] = pd.to_datetime(df["order_purchase_timestamp"], dayfirst=True)

# zostawiamy tylko dostarczone zamowienia (delivered) i usuwamy wiersze bez daty
df = df[df["order_status"] == "delivered"].dropna(subset=["order_purchase_timestamp"])

# wycinamy sama date z timestampa (bez godziny), potrzebne do agregacji dziennej
df["ds"] = df["order_purchase_timestamp"].dt.date

# uzupelniamy brakujace kategorie jako unknown zamiast NaN
df["product_category_name_english"] = df["product_category_name_english"].fillna("unknown")

# usuwamy niepotrzebna kolumne order_status - juz jej nie potrzebujemy
df = df.drop(columns=["order_status"])

print("Shape:", df.shape)
print(df.head())

Shape: (109046, 7)
                           order_id                        product_id   price  \
0  00010242fe8c5a6d1ba2dd792cb16214  4244733e06e7ecb4970a6e2683c13e61   58.90   
1  a3de74fbe2db7081a1c0262272cd4283  601a360bd2a916ecef0e88de72a6531a  129.99   
2  130898c0987d1801452a8ed92a670612  4244733e06e7ecb4970a6e2683c13e61   55.90   
3  d2f2458326a050b121e7acd0e3506d8d  fd1976772549ed5b4aed965293122ac8  279.90   
4  532ed5e14e24ae1f0d735b91524b98b9  4244733e06e7ecb4970a6e2683c13e61   64.90   

  order_purchase_timestamp product_category_name  \
0      2017-09-13 08:59:00            cool_stuff   
1      2017-05-02 22:36:00            cool_stuff   
2      2017-06-28 11:52:00            cool_stuff   
3      2017-05-31 11:02:00            perfumaria   
4      2018-05-18 10:25:00            cool_stuff   

  product_category_name_english          ds  
0                    cool_stuff  2017-09-13  
1                    cool_stuff  2017-05-02  
2                    cool_stuff  2017-06-28

Prognoza per kategoria produktu

In [3]:
# sprawdzamy ile zamowien ma kazda kategoria
# Prophet potrzebuje min. kilkudziesieciu punktow danych zeby model mial sens
category_counts = df.groupby("product_category_name_english").size().sort_values(ascending=False)
print(f"Kategorii ogołem: {len(category_counts)}")
print(category_counts)

Kategorii ogołem: 72
product_category_name_english
bed_bath_table               10839
health_beauty                 9341
sports_leisure                8323
furniture_decor               8080
computers_accessories         7578
                             ...  
arts_and_craftmanship           24
la_cuisine                      14
cds_dvds_musicals               12
fashion_childrens_clothes        7
security_and_services            2
Length: 72, dtype: int64


In [4]:
# filtrujemy tylko kategorie z min. 100 zamowieniami - za malo danych = zly model
MIN_ORDERS = 100
valid_categories = category_counts[category_counts >= MIN_ORDERS].index.tolist()
print(f"Kategorii z >= {MIN_ORDERS} zamowieniami: {len(valid_categories)}")
print(valid_categories)

Kategorii z >= 100 zamowieniami: 54
['bed_bath_table', 'health_beauty', 'sports_leisure', 'furniture_decor', 'computers_accessories', 'housewares', 'watches_gifts', 'telephony', 'garden_tools', 'auto', 'toys', 'cool_stuff', 'perfumery', 'baby', 'electronics', 'stationery', 'fashion_bags_accessories', 'pet_shop', 'office_furniture', 'unknown', 'consoles_games', 'luggage_accessories', 'construction_tools_construction', 'home_appliances', 'small_appliances', 'musical_instruments', 'home_construction', 'books_general_interest', 'food', 'furniture_living_room', 'home_confort', 'audio', 'drinks', 'market_place', 'construction_tools_lights', 'air_conditioning', 'kitchen_dining_laundry_garden_furniture', 'food_drink', 'industry_commerce_and_business', 'books_technical', 'fashion_shoes', 'fixed_telephony', 'costruction_tools_garden', 'home_appliances_2', 'agro_industry_and_commerce', 'computers', 'signaling_and_security', 'art', 'construction_tools_safety', 'christmas_supplies', 'fashion_underw

In [7]:
# funkcja trenujaca model Prophet dla jednej kategorii
def forecast_category(df, category, periods=30):
    sub = df[df["product_category_name_english"] == category].copy()
    daily_cat = sub.groupby("ds").size().reset_index(name="y")
    daily_cat["ds"] = pd.to_datetime(daily_cat["ds"])

    model = Prophet()
    model.fit(daily_cat)

    future = model.make_future_dataframe(periods=periods)
    forecast = model.predict(future)
    return forecast, daily_cat

In [8]:
# trenujemy modele dla wszystkich kategorii i zapisujemy wyniki w slowniku
# moze to chwile potrwac - Prophet trenuje osobny model dla kazdej kategorii
all_forecasts = {} # klucz: nazwa kategorii, wartosc: (forecast, daily)

for i, cat in enumerate(valid_categories):
    print(f"[{i+1}/{len(valid_categories)}] Trenowanie: {cat}")
    forecast, daily = forecast_category(df, cat)
    all_forecasts[cat] = (forecast, daily)

print("Gotowe! Wszystkie modele wytrenowane.")

21:46:45 - cmdstanpy - INFO - Chain [1] start processing


[1/54] Trenowanie: bed_bath_table


21:46:45 - cmdstanpy - INFO - Chain [1] done processing
21:46:45 - cmdstanpy - INFO - Chain [1] start processing


[2/54] Trenowanie: health_beauty


21:46:46 - cmdstanpy - INFO - Chain [1] done processing
21:46:46 - cmdstanpy - INFO - Chain [1] start processing


...

[53/54] Trenowanie: costruction_tools_tools


21:47:00 - cmdstanpy - INFO - Chain [1] done processing
21:47:01 - cmdstanpy - INFO - Chain [1] start processing


[54/54] Trenowanie: furniture_bedroom


21:47:01 - cmdstanpy - INFO - Chain [1] done processing


Gotowe! Wszystkie modele wytrenowane.


In [20]:
### budujemy jeden wykres plotly ze wszystkimi kategoriami
# kazda kategoria to osobny zestaw traces (ukrytych), dropdown przelacza widocznosc

fig = go.Figure()

# dodajemy traces dla kazdej kategorii - na poczatku wszystkie ukryte (visible=False)
# pierwsza kategoria bedzie widoczna od razu (visible=True)
for i, cat in enumerate(valid_categories):
    forecast, daily = all_forecasts[cat]
    is_visible = (i == 0) # tylko pierwsza kategoria widoczna na starcie

    # dane historyczne - czarne punkty
    fig.add_trace(go.Scatter(
        x=daily["ds"], y=daily["y"],
        mode="markers",
        marker=dict(color="black", size=3),
        name="Actual",
        visible=is_visible
    ))

    # prognoza - niebieska linia
    fig.add_trace(go.Scatter(
        x=forecast["ds"], y=forecast["yhat"],
        mode="lines",
        line=dict(color="royalblue"),
        name="Forecast (yhat)",
        visible=is_visible
    ))

    # przedzial niepewnosci - jasnoniebieski obszar
    fig.add_trace(go.Scatter(
        x=pd.concat([forecast["ds"], forecast["ds"].iloc[::-1]]),
        y=pd.concat([forecast["yhat_upper"], forecast["yhat_lower"].iloc[::-1]]),
        fill="toself",
        fillcolor="rgba(65,105,225,0.15)",
        line=dict(color="rgba(255,255,255,0)"),
        name="Uncertainty",
        visible=is_visible
    ))

# budujemy liste przyciskow dropdown - kazdy przycisk pokazuje 3 traces jednej kategorii
# i ukrywa pozostale (kazda kategoria = 3 traces: actual, forecast, uncertainty)
TRACES_PER_CAT = 3
buttons = []

for i, cat in enumerate(valid_categories):
    # tworzymy liste True/False dla kazdego trace
    # True tylko dla 3 traces aktualnie wybranej kategorii
    visibility = [False] * (len(valid_categories) * TRACES_PER_CAT)
    visibility[i * TRACES_PER_CAT] = True     # actual
    visibility[i * TRACES_PER_CAT + 1] = True  # forecast
    visibility[i * TRACES_PER_CAT + 2] = True  # uncertainty

    buttons.append(dict(
        label=cat,
        method="update",
        args=[
            {"visible": visibility},
            {"title": f"Demand Forecast — {cat}"}
        ]
    ))

# dodajemy dropdown do layoutu wykresu
fig.update_layout(
    title=f"Demand Forecast — {valid_categories[0]}",
    xaxis_title="Date",
    yaxis_title="Orders per day",
    height=550,
    margin=dict(t=120), # miejsce na tytul i dropdown zeby sie nie nakladaly
    updatemenus=[dict(
        active=0,
        buttons=buttons,
        x=0.0,
        xanchor="left",
        y=1.0,
        yanchor="bottom",
        bgcolor="white",
        bordercolor="gray"
    )]
)

fig.show()

In [21]:
# budujemy interaktywny wykres sezonowosci miesiecznej z dropdownem
# dla kazdej kategorii liczymy srednia liczbe zamowien per miesiac
# pozwala zobaczyc w ktorych miesiacach popyt jest najwyzszy

MONTH_NAMES = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
               "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

fig_seasonal = go.Figure()

for i, cat in enumerate(valid_categories):
    _, daily = all_forecasts[cat]
    is_visible = (i == 0) # tylko pierwsza kategoria widoczna na starcie

    # dodajemy kolumne miesiac i liczymy srednia zamowien per miesiac
    daily_copy = daily.copy()
    daily_copy["month"] = pd.to_datetime(daily_copy["ds"]).dt.month
    monthly_avg = daily_copy.groupby("month")["y"].mean()

    # uzupelniamy brakujace miesiace zerami (np. kategoria bez sprzedazy w danym miesiacu)
    monthly_avg = monthly_avg.reindex(range(1, 13), fill_value=0)

    fig_seasonal.add_trace(go.Bar(
        x=MONTH_NAMES,
        y=monthly_avg.values,
        name=cat,
        marker_color="royalblue",
        visible=is_visible
    ))

# budujemy dropdown - analogicznie jak w wykresie prognozy
# ale tutaj kazda kategoria = 1 trace (nie 3) wiec prostszy
buttons_seasonal = []
for i, cat in enumerate(valid_categories):
    visibility = [False] * len(valid_categories)
    visibility[i] = True

    buttons_seasonal.append(dict(
        label=cat,
        method="update",
        args=[
            {"visible": visibility},
            {"title": f"Monthly Seasonality — {cat}"}
        ]
    ))

fig_seasonal.update_layout(
    title=f"Monthly Seasonality — {valid_categories[0]}",
    xaxis_title="Month",
    yaxis_title="Avg orders per day",
    height=500,
    margin=dict(t=120),
    updatemenus=[dict(
        active=0,
        buttons=buttons_seasonal,
        x=0.0,
        xanchor="left",
        y=1.0,
        yanchor="bottom",
        bgcolor="white",
        bordercolor="gray"
    )]
)

fig_seasonal.show()

In [22]:
# eksport prognoz dla wszystkich kategorii do jednego pliku csv
all_rows = []
for cat, (forecast, _) in all_forecasts.items():
    forecast["category"] = cat
    all_rows.append(forecast[["ds", "yhat", "yhat_lower", "yhat_upper", "category"]])

pd.concat(all_rows).to_csv("forecast_categories.csv", index=False)


# eksport wykresu do interaktywnego pliku html - mozna otworzyc w przegladarce bez Pythona
fig.write_html("forecast_categories.html")
fig_seasonal.write_html("seasonality_categories.html")
